## Structured output

Models casn be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

## Pydantic

pydantic models provide the richest feature set with field validation, description and nested structures.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

In [38]:
from sqlalchemy import desc
# pydantic will help you build structured output

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description='The title of the movie')
    year:int=Field(description='The year the movie was released')
    director:str=Field(description='The director of the movie')
    rating:float=Field(description='The movies rating out of 10')

In [39]:
model_with_structure = model.with_structured_output(Movie)
print(model_with_structure)

first=_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D81DFBAFD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D81DFBBD90>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': '

In [40]:
response = model_with_structure.invoke('Provide details about the movie inception')
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output  alongside parsed structure

In [41]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        ...,
        description="The title of the movie"
    )

    year: int = Field(
        ...,
        description="The year the movie was released"
    )

    director: str = Field(
        ...,
        description="The director of the movie"
    )
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke('Provide details about the moview Inception')
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about the moview Inception". Likely they want details like director, title, year. Could use function Movie. Provide details. Use function call with director Christopher Nolan, title Inception, year 2010.', 'tool_calls': [{'id': 'fc_29f83cf1-cebc-4d3d-8b87-7847ea2ddafc', 'function': {'arguments': '{"director":"Christopher Nolan","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 147, 'total_tokens': 242, 'completion_time': 0.202482176, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.027049888, 'prompt_tokens_details': None, 'queue_time': 0.2144904, 'total_time': 0.229532064}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r

## Nested Structure

In [42]:
from typing import List
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: List[Actor]
    genres: List[str]
    budget: float | None = Field(None, description='Budget in Million Dollars')

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke('Provide details about the movie Inception')
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)

## TypedDic

TypedDict Provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [43]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, ..., 'The title of the movie']
    year: Annotated[int, ..., 'The year the movie was released']
    director: Annotated[str, ..., 'The director of the movie']
    rating: Annotated[float, ..., 'The movie rating out of 10']

model_with_typedict = model.with_structured_output(MovieDict)
response= model_with_typedict.invoke('Provide the details of the movie Avengers')
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [44]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    cast: Annotated[list[Actor], "The actors in the movie and their roles"]
    genres: Annotated[list[str], "The movie genres"]
    budget: Annotated[
        float | None,
        "Budget in Million Dollars"
    ]

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Fast and Furious"
)

print(response)

{'budget': 38, 'cast': [{'name': 'Vin Diesel', 'role': 'Dominic Toretto'}, {'name': 'Paul Walker', 'role': "Brian O'Conner"}, {'name': 'Michelle Rodriguez', 'role': 'Letty Ortiz'}, {'name': 'Jordana Brewster', 'role': 'Mia Toretto'}, {'name': 'Rick Yune', 'role': 'Han Lue'}, {'name': 'Chad Lindberg', 'role': 'Jesse'}], 'genres': ['Action', 'Crime', 'Thriller'], 'title': 'The Fast and the Furious', 'year': 2001}


In [45]:
# profile = this is what the model actually support

model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [46]:
## Huggingface

"""import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3.5-35B-A3B",
    task="conversational",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    max_new_tokens=1024,
    temperature=0.6,
)

model = ChatHuggingFace(llm=llm)""

_IncompleteInputError: incomplete input (3674920234.py, line 3)

In [47]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

## DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [ ]:
# using pydantic and agent
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: Francis Royce, francisroyce12@gmail.com, (+234) 8162196793"
        }
    ]
})

print(result["structured_response"])


name='Francis Royce' email='francisroyce12@gmail.com' phone='(+234) 8162196793'


In [52]:
# using typedict and agent

from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: Francis Royce, francisroyce12@gmail.com, (+234) 8162196793"
        }
    ]
})

print(result["structured_response"])


{'name': 'Francis Royce', 'email': 'francisroyce12@gmail.com', 'phone': '(+234) 8162196793'}


In [53]:
# dataclass

from dataclasses import  dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: Francis Royce, francisroyce12@gmail.com, (+234) 8162196793"
        }
    ]
})

print(result["structured_response"])


ContactInfo(name='Francis Royce', email='francisroyce12@gmail.com', phone='(+234) 8162196793')
